# PBC-Analysis — 季度工作叙述 → 固定 JSON → 绩效接口

**流程**：读取文本 → LLM 结构化 → 校验 → 保存 `outputs/runtime/pbc_payload.json` →（可选）调用制订/反馈 URL。

运行前请在项目根目录配置 `.env`（可复制 `.env.example`）。

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import dotenv_values


def _resolve_project_root() -> Path:
    """不依赖「从哪启动 jupyter」：向上找含 main.ipynb 的本项目目录。"""
    cwd = Path.cwd().resolve()
    if (cwd / "main.ipynb").is_file() and (cwd / "src" / "extract.py").is_file():
        return cwd
    alt = cwd / "Co-creation-projects" / "TenSon19920106-PBC-Analysis"
    if (alt / "main.ipynb").is_file():
        return alt
    for p in cwd.parents:
        hit = p / "Co-creation-projects" / "TenSon19920106-PBC-Analysis" / "main.ipynb"
        if hit.is_file():
            return hit.parent
    return cwd


ROOT = _resolve_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

_env_path = ROOT / ".env"
if _env_path.is_file():
    # 直接从文件写入 os.environ，彻底覆盖系统里残留的 LLM_MODEL_ID（如 GLM-5.1）
    for _k, _v in dotenv_values(_env_path).items():
        if _v is not None and str(_v).strip() != "":
            os.environ[_k] = str(_v).strip()
else:
    print("警告：未找到", _env_path)

print("项目根目录:", ROOT)
print("已加载 .env:", _env_path)
print("当前 LLM_MODEL_ID =", repr(os.getenv("LLM_MODEL_ID")))
print("当前 LLM_BASE_URL =", repr(os.getenv("LLM_BASE_URL")))

项目根目录: C:\Users\80025013\hello-agents\Co-creation-projects\TenSon19920106-PBC-Analysis
已加载 .env: C:\Users\80025013\hello-agents\Co-creation-projects\TenSon19920106-PBC-Analysis\.env
当前 LLM_MODEL_ID = 'glm-5'
当前 LLM_BASE_URL = 'https://open.bigmodel.cn/api/paas/v4/'


In [2]:
from hello_agents import HelloAgentsLLM, SimpleAgent, ToolRegistry

from src.extract import extract_pbc_payload, fill_submitted_at_if_missing
from src.io_utils import load_text_file
from src.schema import validate_pbc_payload
from src.pbc_tools import ReadWorkSummaryTool, SubmitPbcPlanningTool, SubmitPbcFeedbackTool

print("依赖加载完成")

依赖加载完成


## 1.1 先在下一格填写三个必填参数

请在**下面的代码单元格**里填写：

- `GH`：工号（字符串，例如 `"80025013"`）
- `KHNF_INPUT`：可填年份（如 `2026`）或编码（如 `3`）
- `KHJD_INPUT`：可填 `"第一季度"/"Q1"/0`（或其它季度/年度）

注意：写在这里（Markdown）不会被执行，必须写在代码格里。

In [3]:
from src.normalize import normalize_khnf, normalize_khjd

# ===== 必填参数（请修改这里） =====
GH = "80025013"        # 工号：必须是字符串
KHNF_INPUT = 2026       # 可填 2026 或 3（表示 2026）
KHJD_INPUT = "Q1"      # 可填 "第一季度" / "Q1" / 0

# ===== 校验与规范化 =====
if not str(GH).strip():
    raise ValueError("请先填写 GH（工号）")
KHNF = normalize_khnf(KHNF_INPUT)
KHJD = normalize_khjd(KHJD_INPUT)
print("已解析参数 =>", {"gh": GH, "khnf": KHNF, "khjd": KHJD})

# ===== 文本输入（二选一） =====
USER_TEXT = ""  # 若为空则使用下方文件
INPUT_FILE = "sample_quarterly_work.txt"

if USER_TEXT.strip():
    raw = USER_TEXT.strip()
else:
    raw = load_text_file(ROOT / "data" / INPUT_FILE)

print(raw[:800] + ("..." if len(raw) > 800 else ""))

已解析参数 => {'gh': '80025013', 'khnf': 3, 'khjd': 0}
2026 年第一季度工作总结（示例）

本季度我主要负责订单中台与结算链路的稳定性治理。2 月完成了一次核心链路压测，将峰值下单失败率从约 0.8% 降到 0.15%。
3 月牵头梳理了与财务对账的差异清单，推动 4 个历史遗留问题闭环。

亮点：上线灰度发布策略，回滚时间从平均 12 分钟缩短到 3 分钟内。
困难：第三方支付渠道偶发超时，缺少统一熔断与降级策略，已排期下季度改造。

下季度计划：完成支付渠道熔断与配额治理；推进对账自动化报表一期。

需要支持：希望协调财务同事每周固定半小时对账例会，便于快速确认差异。


## 2. LLM 结构化 → 校验 → 落盘

In [4]:
llm = HelloAgentsLLM(
    model=os.getenv("LLM_MODEL_ID"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
    timeout=int(os.getenv("LLM_TIMEOUT", "120")),
)
print("HelloAgentsLLM 使用 model =", repr(llm.model))

payload = extract_pbc_payload(llm, raw, gh=GH, khnf=KHNF, khjd=KHJD)
payload = fill_submitted_at_if_missing(payload)

ok, errs = validate_pbc_payload(payload)
assert ok, errs

out_dir = ROOT / "outputs" / "runtime"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "pbc_payload.json"
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写入:", out_path)
print(json.dumps(payload, ensure_ascii=False, indent=2)[:2000])

HelloAgentsLLM 使用 model = 'glm-5'
✅ Trace 已保存:
   JSONL: memory\traces\trace-s-20260413-162941-cf62.jsonl
   HTML:  memory\traces\trace-s-20260413-162941-cf62.html
已写入: C:\Users\80025013\hello-agents\Co-creation-projects\TenSon19920106-PBC-Analysis\outputs\runtime\pbc_payload.json
{
  "gh": "80025013",
  "khnf": 3,
  "khjd": 0,
  "dllzb": [
    {
      "khmd": "提升核心链路稳定性",
      "wd": 2,
      "kpi": "峰值下单失败率",
      "mb": "降至0.15%以内",
      "ly": 3,
      "qz": 40.0,
      "jffs": "目标达成率"
    },
    {
      "khmd": "解决历史遗留问题",
      "wd": 2,
      "kpi": "历史遗留问题闭环数量",
      "mb": "完成4个",
      "ly": 3,
      "qz": 20.0,
      "jffs": "计数法"
    }
  ],
  "dxlzb": [
    {
      "khmd": "保障系统稳定性与发布效率",
      "zdgz": "订单中台与结算链路稳定性治理；上线灰度发布策略，缩短回滚时间",
      "ly": 3,
      "qz": 40.0,
      "jffs": "关键事件评分法"
    }
  ],
  "grcz": [
    {
      "dtgnl": "熔断与降级策略设计能力",
      "jhtscs": "完成支付渠道熔断与配额治理项目",
      "sxzy": "原文未提及",
      "jhwcsj": "2026-06-30"
    }
  ]
}


## 2.5 信息不足时：提示用户补充后再生成

**思路**：先用规则检查 `payload` 是否有明显缺口 → 打印追问清单 → 用户在下方填写 `USER_SUPPLEMENT` → 再跑下一格把补充拼回原文并**重新结构化**。

以后做成网页时，把「追问清单」当表单字段展示即可；逻辑仍可用本节的 `list_information_gaps` / `merge_raw_with_supplement`。

In [5]:
from src.completeness import (
    list_information_gaps,
    format_gaps_for_user,
    merge_raw_with_supplement,
    suggest_follow_up_questions,
)

_gaps = list_information_gaps(payload)
if not _gaps:
    print("关键信息检查：未发现明显缺口（规则层），可继续后续流程。")
else:
    print("=== 请让员工补充以下信息（复制到邮件/表单或让用户在下方填写）===\n")
    print(format_gaps_for_user(_gaps))
    print("\n--- LLM 生成的追问（可选）---\n")
    print(suggest_follow_up_questions(llm, raw_text=raw, gaps=_gaps)[:3000])

# 用户补充内容写在这里后，运行本格下半段以更新 payload 与 json 文件
USER_SUPPLEMENT = ""

if USER_SUPPLEMENT.strip():
    raw = merge_raw_with_supplement(raw, USER_SUPPLEMENT.strip())
    payload = extract_pbc_payload(llm, raw, gh=GH, khnf=KHNF, khjd=KHJD)
    payload = fill_submitted_at_if_missing(payload)
    ok, errs = validate_pbc_payload(payload)
    assert ok, errs
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n已根据补充内容更新:", out_path)
    _gaps2 = list_information_gaps(payload)
    print("补充后仍存在的规则缺口:" if _gaps2 else "补充后规则检查通过。", format_gaps_for_user(_gaps2) if _gaps2 else "")

=== 请让员工补充以下信息（复制到邮件/表单或让用户在下方填写）===

1. 工作总述过短（少于 40 字），请补充职责范围、主要产出或协作方。
2. 亮点成果过少或过于笼统，请补充可量化指标（百分比、次数、金额、周期等）。
3. 未描述困难或风险；若本季度无显著困难，可简要写「无重大阻塞」并说明原因。
4. 下季度计划为空，请至少写 1～2 条可执行计划。
5. 所需支持为空；若不需要支持，可写「暂不需要额外支持」。

--- LLM 生成的追问（可选）---

✅ Trace 已保存:
   JSONL: memory\traces\trace-s-20260413-163058-8ad2.jsonl
   HTML:  memory\traces\trace-s-20260413-163058-8ad2.html
1. 请扩充工作总述，补充具体的职责范围、主要产出成果及协作方信息。
2. 请进一步量化亮点成果，补充具体的百分比、金额或周期等数据指标。
3. 请补充本季度遇到的困难或风险描述；若无显著困难，请说明原因。
4. 请补充下季度工作计划，列出至少 1～2 条具体的可执行事项。
5. 请补充所需支持内容，若暂无需求请注明“暂不需要额外支持”。


## 3.（可选）带工具的编排智能体

演示「先读文件、再提交」的工具链：需正确配置环境变量；未配置 URL 时工具返回干跑说明。

In [6]:
registry = ToolRegistry()
registry.register_tool(ReadWorkSummaryTool(ROOT / "data"))
registry.register_tool(SubmitPbcPlanningTool())
registry.register_tool(SubmitPbcFeedbackTool())

orchestrator = SimpleAgent(
    name="PBC流程编排",
    llm=llm,
    system_prompt=(
        "你是绩效系统助手。若用户要提交 JSON，请使用 submit_pbc_planning 或 submit_pbc_feedback，"
        "参数 payload_json 必须是完整 JSON 字符串。若用户要读总结文件，使用 read_work_summary_file。"
        "不要编造 URL。"
    ),
    tool_registry=registry,
)

demo_json = json.dumps(payload, ensure_ascii=False)
q = f"请将下列 PBC JSON 提交到绩效制订接口（若未配置请说明干跑结果）：\n{demo_json[:4000]}"
print(orchestrator.run(q)[:3000])

✅ 工具 'read_work_summary_file' 已注册。
✅ 工具 'submit_pbc_planning' 已注册。
✅ 工具 'submit_pbc_feedback' 已注册。
✅ 工具 'Skill' 已注册。
✅ 工具 'Task' 已注册。
✅ 工具 'TodoWrite' 已注册。
✅ 工具 'DevLog' 已注册。
✅ Trace 已保存:
   JSONL: memory\traces\trace-s-20260413-163236-2c58.jsonl
   HTML:  memory\traces\trace-s-20260413-163236-2c58.html
## 提交结果

**状态：干跑模式（未实际提交）**

### 说明
由于未配置 `PBC_PLANNING_URL` 环境变量，系统未发起实际的 HTTP 请求，仅进行了干跑验证。

### 载荷验证结果
✅ JSON 格式正确  
✅ 载荷结构完整，包含以下顶层键：
- `gh` - 工号
- `khnf` - 考核年份
- `khjd` - 考核阶段
- `dllzb` - 定量类指标（2项）
- `dxlzb` - 定性类指标（1项）
- `grcz` - 个人成长（1项）

### PBC 内容摘要
| 类别 | 考核目标 | 权重 |
|------|----------|------|
| 定量指标 | 提升核心链路稳定性 | 40% |
| 定量指标 | 解决历史遗留问题 | 20% |
| 定性指标 | 保障系统稳定性与发布效率 | 40% |
| 个人成长 | 熔断与降级策略设计能力 | - |

---

如需实际提交，请先配置 `PBC_PLANNING_URL` 环境变量，然后重新执行提交操作。
